In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.makedirs("./outputs/new_figs", exist_ok=True)

## Importing

In [ ]:
import pandas as pd
from scripts.data_utils import cut_obs_df
import matplotlib.pyplot as plt

import matplotlib.cm as cm
import numpy as np

from scipy.spatial.distance import euclidean, cityblock, jensenshannon
from scipy.stats import wasserstein_distance

import seaborn as sns

import osmnx as ox

## Helper functions

In [ ]:
# Color palettes defined outside the function for reuse
WEEKDAY_STYLE = {
    "color": "#4361EE",
    "edgecolor": "#2D47C9",
    "mediancolor": "#1B2FA8",
    "alpha": 0.7,
    "label": "Weekdays",
}

WEEKEND_STYLE = {
    "color": "#F72585",
    "edgecolor": "#C41E6A",
    "mediancolor": "#5A108B",
    "alpha": 0.7,
    "label": "Weekend",
}


def _plot_profile(hourly_long, style, ylabel, save_path=None):
    """Helper to plot a boxplot profile."""
    fig, ax = plt.subplots(figsize=(10, 5))

    hours = sorted(hourly_long["hour"].unique())
    data_by_hour = [hourly_long.loc[hourly_long["hour"] == h, "value"].values for h in hours]

    bp = ax.boxplot(
        data_by_hour,
        positions=hours,
        widths=0.6,
        patch_artist=True,
        boxprops=dict(facecolor=style["color"], alpha=style["alpha"], edgecolor=style["edgecolor"]),
        medianprops=dict(color=style["mediancolor"], linewidth=2),
        showmeans=False,
        showfliers=False,
        whiskerprops=dict(color=style["edgecolor"]),
        capprops=dict(color=style["edgecolor"]),
        label=style["label"],
    )

    ax.set_xlabel("Hour of Day")
    ax.set_ylabel(ylabel)
    ax.set_xticks(hours)
    ax.set_xticklabels([f"{h:02d}:00" for h in hours], rotation=45, ha="right", fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.3, axis="y")
    ax.legend(handles=[bp["boxes"][0]])
    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, format="pdf", bbox_inches="tight")

    plt.show()


def plot_weekday_vs_weekend_profile(
    df,
    aggr_method="mean",
    normalize=False,
    save_path_weekday=None,
    save_path_weekend=None,
):
    # 1. Resample to hourly FIRST, before any splitting
    hourly = df.resample("h").sum()

    value_cols = [c for c in hourly.columns]

    if aggr_method == "mean":
        hourly["value"] = hourly[value_cols].mean(axis=1)
    elif aggr_method == "sum":
        hourly["value"] = hourly[value_cols].sum(axis=1)
    else:
        raise ValueError("aggr_method must be 'mean' or 'sum'")

    hourly["hour"] = hourly.index.hour

    # 2. Optionally normalize per day BEFORE splitting
    if normalize:
        daily_totals = hourly.groupby(hourly.index.date)["value"].transform("sum")
        hourly["value"] = hourly["value"] / daily_totals

    # 3. Split into weekday / weekend AFTER resampling and normalization
    hourly_weekday = hourly[hourly.index.weekday < 5][["hour", "value"]].copy()
    hourly_weekend = hourly[hourly.index.weekday >= 5][["hour", "value"]].copy()

    print("Weekday profile:")
    print(hourly_weekday.groupby("hour")["value"].describe())
    print("\nWeekend profile:")
    print(hourly_weekend.groupby("hour")["value"].describe())

    ylabel = "Traffic Volume" if not normalize else "Share of Daily Traffic"

    _plot_profile(hourly_weekday, WEEKDAY_STYLE, ylabel, save_path=save_path_weekday)
    _plot_profile(hourly_weekend, WEEKEND_STYLE, ylabel, save_path=save_path_weekend)

    return hourly_weekday, hourly_weekend

In [ ]:
def compute_hourly_profile(hourly_totals, aggr_method="mean", normalize=False):
    """
    hourly_totals: already-resampled hourly Series (output of resample("h").sum().mean(axis=1))
    """
    grouped = hourly_totals.groupby(hourly_totals.index.hour)

    if aggr_method == "mean":
        mean_profile = grouped.mean()
    elif aggr_method == "sum":
        mean_profile = grouped.sum()
    else:
        raise ValueError("aggr_method must be 'mean' or 'sum'")

    std_profile = grouped.std()

    mean_profile = mean_profile.reindex(range(24), fill_value=0)
    std_profile  = std_profile.reindex(range(24), fill_value=0)

    if normalize:
        total = mean_profile.sum()
        mean_profile = mean_profile / total
        std_profile  = std_profile  / total  # use same total, not post-normalized mean

    return mean_profile, std_profile


def plot_hourly_comparison(df_before, df_during, df_after,
                           aggr_method="mean",
                           normalize=False,
                           save_path=None,
                           title="Hourly Traffic Profile Comparison"):

    # 1. Resample ALL dataframes to hourly FIRST
    hourly_before = df_before.resample("h").sum().mean(axis=1)
    hourly_during = df_during.resample("h").sum().mean(axis=1)
    hourly_after  = df_after.resample("h").sum().mean(axis=1)

    # 2. Then compute profiles from the resampled series
    prof_before, std_before = compute_hourly_profile(hourly_before, aggr_method, normalize)
    prof_during, std_during = compute_hourly_profile(hourly_during, aggr_method, normalize)
    prof_after,  std_after  = compute_hourly_profile(hourly_after,  aggr_method, normalize)

    plt.figure(figsize=(10, 5))

    plt.plot(prof_before.index, prof_before.values, '-o', label="Before Closure")
    plt.fill_between(prof_before.index, prof_before - std_before, prof_before + std_before, alpha=0.2)

    plt.plot(prof_during.index, prof_during.values, '-o', label="During Closure")
    plt.fill_between(prof_during.index, prof_during - std_during, prof_during + std_during, alpha=0.2)

    plt.plot(prof_after.index, prof_after.values, '-o', label="After Closure")
    plt.fill_between(prof_after.index, prof_after - std_after, prof_after + std_after, alpha=0.2)

    plt.xlabel("Hour of Day")
    plt.ylabel("Average Traffic Volume" if not normalize else "Share of Daily Traffic")

    hours = range(24)
    plt.xticks(hours, [f"{h:02d}:00" for h in hours], rotation=45, ha="right", fontsize=9)
    plt.grid(True, alpha=0.3, linestyle='--', axis='y')
    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.show()

    return prof_before, prof_during, prof_after


def compare_distributions(p1, p2):
    """
    p1, p2 are 24-length pandas Series.
    Returns dictionary of distance metrics.
    """
    a = p1.values
    b = p2.values

    a_norm = a / a.sum()
    b_norm = b / b.sum()

    metrics = {
        "Euclidean Distance":    euclidean(a, b),
        "Manhattan Distance":    cityblock(a, b),
        "Wasserstein Distance":  wasserstein_distance(range(24), range(24),
                                                      u_weights=a_norm,
                                                      v_weights=b_norm),
        "Jensen-Shannon Distance": jensenshannon(a_norm, b_norm),
    }

    return metrics

In [ ]:
def plot_hourly_zero_frequency(df, save_path=None):
    """
    df: pandas DataFrame
        - index must be datetime
        - columns = sensors
        - values = measurements
    """

    # Ensure datetime index
    df = df.copy()
    df.index = pd.to_datetime(df.index)

    # Create binary dataframe: 1 if zero, 0 otherwise
    zero_df = (df == 0).astype(int)

    # Group by hour and compute zero frequency per sensor
    hourly_zero_freq = zero_df.groupby(df.index.hour).sum()

    # Melt for seaborn
    melted = hourly_zero_freq.reset_index().melt(
        id_vars='index', var_name='sensor', value_name='zero_freq'
    )
    melted.rename(columns={'index': 'hour'}, inplace=True)

    # Plot
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=melted, x='hour', y='zero_freq', showfliers=False)

    # # Compute mean and std per hour
    # stats = melted.groupby('hour')['zero_freq'].agg(['mean', 'std']).reset_index()

    # # Overlay mean
    # plt.plot(stats['hour'], stats['mean'], color='red', marker='o', label='Mean')

    # # Overlay std as error bars
    # plt.errorbar(
    #     stats['hour'],
    #     stats['mean'],
    #     yerr=stats['std'],
    #     fmt='none',
    #     ecolor='red',
    #     capsize=3,
    #     label='Std'
    # )

    # plt.title('Hourly Zero Frequency Distribution')
    plt.xlabel('Hour of Day')
    plt.ylabel('Zero Frequency')
    
    hours = range(24)
    plt.xticks(
        hours,
        [f"{h:02d}:00" for h in hours],
        rotation=45,
        ha="right",
        fontsize=9
    )
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.3, axis='y')

    if save_path is not None:
        plt.savefig(save_path, format="pdf", bbox_inches="tight")
        
    plt.show()

## Load data

In [ ]:
from scripts.data_utils import merge_data_by_junc
import geopandas as gpd
from scripts.data_utils import from_df_to_geopandas
import pandas as pd

traffic_cams_df_with_dirs = gpd.read_file("./data/prod/pre-process/traffic_cams/gpd/traffic_cam_metadata.shp")
aggregated_timeseries = pd.read_csv("./data/prod/pre-process/traffic_cams/pd_time_series.csv", index_col=0, parse_dates=True)


In [ ]:
aggregated_timeseries.index = pd.to_datetime(aggregated_timeseries.index, utc=True).tz_convert("Europe/Rome")

In [ ]:
traffic_cams_df_with_dirs.crs

## Weekdays vs. Weekends

In [ ]:
prof_weekday, prof_weekend = plot_weekday_vs_weekend_profile(
    aggregated_timeseries,
    aggr_method="mean",
    normalize=False,
    save_path_weekday="./outputs/new_figs/weekday_vs_weekend_profile_weekday.pdf",
    save_path_weekend="./outputs/new_figs/weekday_vs_weekend_profile_weekend.pdf"
)

## School closure

In [ ]:
school_sensors_north = [
    "SITO11-ADABASSANO", 
    "LT_2_009B_LT CAMERINI VS ROTATORIA ANNIBALE DA BASSANO", 
    "LT_2_009A_LT CAMERINI VS ROTATORIA CAVALCAVIA CAMERINI", 
    "LT_2_010B_LT VIANELLO-BUONARROTI VS ROTONDA BUONARROTI", 
    "LT_2_010A_LT VIANELLO-GIUCCIARDINI VS CAVALCAVIA CAMERINI", 
    "LT_2_011B_VIA RENI/VIA DUPRE VS CAVALCAVIA BORGOMAGNO/CENTRO STORICO", 
    "LT_2_011A_VIA RENI/VIA DUPRE VS TANGENZALE/VIGODARZERE"
]

school_sensors_ts = aggregated_timeseries[school_sensors_north]

data_9_11_feb_schools_north = cut_obs_df(school_sensors_ts, 1770591600000, 1770764400000)
data_16_18_feb_schools_north = cut_obs_df(school_sensors_ts, 1771196400000, 1771369200000)
data_23_25_feb_schools_north = cut_obs_df(school_sensors_ts, 1771801200000, 1771974000000)

In [ ]:
prof_before, prof_during, prof_after = plot_hourly_comparison(
    data_9_11_feb_schools_north, 
    data_16_18_feb_schools_north, 
    data_23_25_feb_schools_north, 
    aggr_method="mean", 
    normalize=False,
    title = "Hourly Traffic Profile Comparison (Sensors closed to schools)",
    save_path = "./outputs/new_figs/school_closure_comparison_schools_north.pdf"
)

In [ ]:
data_9_11_feb = cut_obs_df(aggregated_timeseries, 1770591600000, 1770764400000)
data_16_18_feb = cut_obs_df(aggregated_timeseries, 1771196400000, 1771369200000)
data_23_25_feb = cut_obs_df(aggregated_timeseries, 1771801200000, 1771974000000)

In [ ]:
prof_before, prof_during, prof_after = plot_hourly_comparison(
    data_9_11_feb, 
    data_16_18_feb, 
    data_23_25_feb, 
    aggr_method="mean", 
    normalize=False,
    save_path = "./outputs/new_figs/school_closure_comparison_all.pdf",
    title = "Hourly Traffic Profile Comparison (All sensors)"
)

## Zero values distr

In [ ]:
plot_hourly_zero_frequency(aggregated_timeseries, save_path="./outputs/new_figs/hourly_zero_frequency.pdf")

## Plots

In [ ]:
# import pydeck as pdk
# import pandas as pd

# # Sensors as nodes
# sensor_layer = pdk.Layer(
#     "ScatterplotLayer",
#     data=traffic_cams_df_with_dirs,
#     get_position=["LON", "LAT"],
#     get_radius=50,
#     get_fill_color=[255, 100, 0, 200],
# )
# view = pdk.ViewState(latitude=45.40622305006962, longitude=11.876285735508933, zoom=12, pitch=45)
# pdk.Deck(layers=[sensor_layer], initial_view_state=view).to_html("map.html")


In [ ]:
zone_file_path = "./data/prod/pre-process/context/SIT_SEZIONI_2021/SIT_SEZIONI_2021.shp"
pop_file_path = "./data/prod/pre-process/context/SIT_SEZIONI_2021/residenti_x_sezioneISTAT-2021.csv"

zones = gpd.read_file(zone_file_path)
residents = pd.read_csv(pop_file_path, delimiter=";")

zones_filtered = zones[zones['SEZ21'].isin(residents['Sezioni 2021 attribuite'])].copy()
mapping = residents.set_index("Sezioni 2021 attribuite")["Somma - Residenti"]
zones_filtered["POP21"] = zones_filtered["SEZ21"].map(mapping).fillna(zones_filtered["POP21"])

census_gdf = zones_filtered

df_pois = pd.read_csv("./data/prod/pre-process/context/filtered_grouped_poi.csv", index_col=0)
gdf_pois = gpd.GeoDataFrame(
    df_pois.drop(columns=["id"]),
    geometry=gpd.points_from_xy(df_pois.longitude, df_pois.latitude),
    crs="EPSG:4326"
)

In [ ]:
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import box
import pandas as pd

# --- Configuration ---
TARGET_CRS = "EPSG:32632"  # UTM 32N (Metric)
WGS84 = "EPSG:4326"        # Lat/Lon (Standard for OSM)

ROAD_STYLES = {
    "motorway":    ("#e05c5c", 1.4),
    "trunk":       ("#e08c5c", 1.1),
    "primary":     ("#888888", 0.8),
    "secondary":   ("#aaaaaa", 0.6),
    "tertiary":    ("#cccccc", 0.4),
    "residential": ("#dddddd", 0.3),
}

def get_plot_extent(gdfs, padding=500):
    """Calculates a combined bounding box in the target metric CRS."""
    valid_gdfs = [gdf for gdf in gdfs if gdf is not None and not gdf.empty]
    # Ensure all are in the same CRS for the calculation
    geoms = [gdf.to_crs(TARGET_CRS).geometry for gdf in valid_gdfs]
    combined_geoms = pd.concat(geoms)
    minx, miny, maxx, maxy = combined_geoms.total_bounds
    return [minx - padding, maxx + padding, miny - padding, maxy + padding]

def get_infrastructure_by_extent():
    G = ox.load_graphml("./data/prod/pre-process/road_network/osmnx.graphml")
    
    nodes, edges = ox.graph_to_gdfs(G)
    return nodes.to_crs(TARGET_CRS), edges.to_crs(TARGET_CRS)

def plot_base_infrastructure(ax, edges, extent):
    """Plots the road network within the extent."""
    for road_type, (color, lw) in ROAD_STYLES.items():
        subset = edges[edges["highway"].apply(
            lambda x: road_type in x if isinstance(x, list) else x == road_type
        )]
        if not subset.empty:
            subset.plot(ax=ax, color=color, linewidth=lw, zorder=1)
    
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_axis_off()

def generate_census_map(census_gdf, edges, extent, filename):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    plot_base_infrastructure(ax, edges, extent)
    
    census_gdf["POP_DENS"] = census_gdf["POP21"]
    census_gdf.to_crs(TARGET_CRS).plot(
        column="POP_DENS", ax=ax, cmap="YlOrRd",
        legend=True, legend_kwds={"label": "Zone Population", "shrink": 0.4}, zorder=2
    )
    
    # ax.set_title("Population Density Zones", fontsize=14)
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    plt.close()

def generate_poi_map(poi_gdf, edges, extent, filename):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    plot_base_infrastructure(ax, edges, extent)
    
    poi_gdf = poi_gdf.to_crs(TARGET_CRS)
    categories = sorted(poi_gdf["category_level_0"].dropna().unique())
    cmap = plt.get_cmap("tab20", len(categories))
    
    for i, cat in enumerate(categories):
        subset = poi_gdf[poi_gdf["category_level_0"] == cat]
        subset.plot(ax=ax, color=cmap(i), markersize=5, label=cat, zorder=3)
    
    ax.legend(loc="lower right", fontsize=8, title="POIs Categories", ncol=2)
    # ax.set_title("POI Distribution", fontsize=14)
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    plt.close()

def generate_sensor_map(sensors_gdf, edges, extent, filename):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    plot_base_infrastructure(ax, edges, extent)
    
    sensors_gdf.to_crs(TARGET_CRS).plot(
        ax=ax, color="dodgerblue", edgecolor="white",
        linewidth=0.5, markersize=70, marker="^", zorder=5, label="AVI Sensor"
    )
    
    # ax.legend(loc="lower left", fontsize=8)
    # ax.set_title("Traffic Sensor Locations", fontsize=14)
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    plt.close()

In [ ]:
_, road_edges = get_infrastructure_by_extent()

In [ ]:
zoom_extent = get_plot_extent([census_gdf, gdf_pois, traffic_cams_df_with_dirs], padding=1000)

In [ ]:
# 4. Generate the three distinct images
# generate_census_map(census_gdf, road_edges, zoom_extent, "./outputs/new_figs/map_census.pdf")
generate_poi_map(gdf_pois, road_edges, zoom_extent, "./outputs/new_figs/map_pois.pdf")
# generate_sensor_map(traffic_cams_df_with_dirs, road_edges, zoom_extent, "./outputs/new_figs/map_sensors.pdf")

print("Maps generated successfully.")

In [ ]:
# All 95 sensors distribution plot

traffic_cam_95 = gpd.read_file("./preprocess/veh_counts_95_aug.geojson")

In [ ]:
generate_sensor_map(traffic_cam_95, road_edges, zoom_extent, "./outputs/new_figs/map_sensors_all_95.pdf")

In [ ]:
junc_aggr_cams = gpd.read_file("./data/prod/pre-process/traffic_cams_by_junc/gpd/traffic_cam_metadata_by_junc.shp")

generate_sensor_map(junc_aggr_cams, road_edges, zoom_extent, "./outputs/new_figs/map_sensors_by_junc.pdf")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pandas as pd
import geopandas as gpd

def generate_junction_residual_map(
    residuals_df: pd.DataFrame,
    junctions_gdf: gpd.GeoDataFrame,
    edges: gpd.GeoDataFrame,
    extent: list,
    filename: str,
    junction_id_col: str = "junction_id",  # column in junctions_gdf matching residuals_df index
):
    # --- Merge residuals into junctions ---
    junctions = junctions_gdf.copy().to_crs(TARGET_CRS)
    junctions = junctions.merge(
        residuals_df[["normalized_residual"]],
        left_on=junction_id_col,
        right_index=True,
        how="inner",
    )

    # --- Colormap: diverging, centered at 0 ---
    cmap = plt.get_cmap("RdYlGn")          # red = negative, green = positive
    norm = mcolors.TwoSlopeNorm(
        vmin=junctions["normalized_residual"].min(),
        vcenter=0.0,
        vmax=junctions["normalized_residual"].max(),
    )

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    plot_base_infrastructure(ax, edges, extent)

    junctions.plot(
        ax=ax,
        column="normalized_residual",
        cmap=cmap,
        norm=norm,
        markersize=40,
        marker="o",
        edgecolor="white",
        linewidth=0.4,
        zorder=5,
        legend=False,
    )

    # --- Colorbar ---
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.01)
    cbar.set_label("Normalized residual\n(inflow − outflow)", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    ax.set_axis_off()
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    plt.close()

In [ ]:
junctions_gdf = gpd.read_file("./data/prod/pre-process/traffic_cams_by_junc/gpd/traffic_cam_metadata_by_junc.shp")
junctions_gdf

In [ ]:

residuals_df = pd.read_csv("./data/prod/pre-process/plate_hash/flow_residuals.csv", index_col="junctions")
# nodes, edges = get_infrastructure_by_extent()
# extent = get_plot_extent([nodes, junctions_gdf])

generate_junction_residual_map(
    residuals_df=residuals_df,
    junctions_gdf=junctions_gdf,
    edges=road_edges,
    extent=zoom_extent,
    filename="./outputs/new_figs/junction_residuals.pdf",
    junction_id_col="id",   # adjust to match your GeoDataFrame
)

## Traffic flow statistics

### Avg. travel times

In [ ]:
def plot_hourly_travel_time_profile(
    avg_travel_times_hourly: pd.DataFrame,
    save_path_weekday=None,
    save_path_weekend=None,
):
    df = avg_travel_times_hourly.copy()
    df['hour'] = df['time_bin'].dt.hour
    df['day_type'] = df['time_bin'].dt.dayofweek.map(
        lambda d: 'Weekend' if d >= 5 else 'Weekdays'
    )

    def _plot_travel_time_profile(subset, style, save_path=None):
        fig, ax = plt.subplots(figsize=(10, 5))

        hours = sorted(subset['hour'].unique())
        data_by_hour = [subset.loc[subset['hour'] == h, 'mean'].values for h in hours]

        bp = ax.boxplot(
            data_by_hour,
            positions=hours,
            widths=0.6,
            patch_artist=True,
            boxprops=dict(facecolor=style['color'], alpha=style['alpha'], edgecolor=style['edgecolor']),
            medianprops=dict(color=style['mediancolor'], linewidth=2),
            showmeans=False,
            showfliers=False,
            whiskerprops=dict(color=style['edgecolor']),
            capprops=dict(color=style['edgecolor']),
            label=style['label'],
        )

        ax.set_xlabel("Hour of Day")
        ax.set_ylabel("Travel Time (seconds)")
        hours_range = range(24)
        ax.set_xticks(hours_range)
        ax.set_xticklabels([f"{h:02d}:00" for h in hours_range], rotation=45, ha="right", fontsize=9)
        ax.grid(True, linestyle="--", alpha=0.3, axis='y')
        ax.legend(handles=[bp['boxes'][0]])
        fig.tight_layout()

        if save_path is not None:
            fig.savefig(save_path, format="pdf", bbox_inches="tight")

        plt.show()

    for day_type, style in {'Weekdays': WEEKDAY_STYLE, 'Weekend': WEEKEND_STYLE}.items():
        subset = df[df['day_type'] == day_type][['hour', 'mean']].copy()
        save_path = save_path_weekday if day_type == 'Weekdays' else save_path_weekend
        _plot_travel_time_profile(subset, style, save_path=save_path)

In [ ]:
avg_travel_times_hourly = pd.read_csv("./data/prod/pre-process/plate_hash_by_junc/avg_travel_time_hour.csv", index_col=0, parse_dates=["time_bin"])

In [ ]:
avg_travel_times_hourly["time_bin"] = pd.to_datetime(avg_travel_times_hourly["time_bin"].values, utc=True).tz_convert("Europe/Rome")

In [ ]:
avg_travel_times_hourly_ = avg_travel_times_hourly[(avg_travel_times_hourly["count"] > 3) & (avg_travel_times_hourly["mean"] > 10)]

In [ ]:
len(avg_travel_times_hourly_) / len(avg_travel_times_hourly)

In [ ]:
plot_hourly_travel_time_profile(
    avg_travel_times_hourly_, 
    save_path_weekday="./outputs/new_figs/travel_time_profile_weekday.pdf", 
    save_path_weekend="./outputs/new_figs/travel_time_profile_weekend.pdf"
)

## POIs download example

This cell shows the code used to download and pre-process the POIs dataset saved in ./data/prod/pre-process/context/filtered_grouped_poi.csv

In [ ]:
import numpy as np

max_lat, min_lat = traffic_cams_df_with_dirs['LAT'].max(), traffic_cams_df_with_dirs['LAT'].min()
max_lon, min_lon = traffic_cams_df_with_dirs['LON'].max(), traffic_cams_df_with_dirs['LON'].min()

expand_m = 1000
expand_lat = expand_m / 111320.0
mean_lat = (max_lat + min_lat) / 2.0
expand_lon = expand_m / (111320.0 * np.cos(np.radians(mean_lat)))


print(f"Latitude range: {min_lat} to {max_lat}")
print(f"Longitude range: {min_lon} to {max_lon}")

In [ ]:
from pyproj import Geod

bbox = (
    min_lon - expand_lon,
    min_lat - expand_lat,
    max_lon + expand_lon,
    max_lat + expand_lat
)

print(bbox)

In [ ]:
! overturemaps download --bbox=11.81755880708736,45.357261888250086,11.94228119291264,45.45622111174991 -f geojson --type=place -o poi.geojson

In [ ]:
import json
import pandas as pd

with open("./poi.geojson", "r") as f:
    geojson_data = json.load(f)
    
records = []
features = geojson_data["features"]
for feat in features:
    props = feat["properties"]
    geom = feat["geometry"]
    record = {
        "id": props.get("id"),
        "name": props.get("names", {}).get("primary", "-"),
        "version": props.get("version"),
        "confidence": props.get("confidence"),
        "primary_category": props.get("categories", {}).get("primary"),
        "longitude": geom["coordinates"][0] if geom and "coordinates" in geom else None,
        "latitude": geom["coordinates"][1] if geom and "coordinates" in geom else None
    }
    records.append(record)


df = pd.DataFrame(records)

In [ ]:
def filter_by_confidence(df, threshold):
    return df[df["confidence"] >= threshold]

threshold = 0.7
filtered_df = filter_by_confidence(df, threshold)
print(f"\nEntries with confidence >= {threshold}:", len(filtered_df))

In [ ]:
# Categories Taxonomy
import ast

def transform_taxonomy(taxonomy_str):
    items = taxonomy_str.strip("[]").split(",")
    items = [item.strip() for item in items if item.strip()]
    return "[" + ",".join(f'"{item}"' for item in items) + "]"


def build_category_mapping(taxonomy_df, level):
    mapping = {}
    for _, row in taxonomy_df.iterrows():
        cat = row["Category code"]
        path = ast.literal_eval(row["Overture Taxonomy"])
        if level < len(path):
            mapping[cat] = path[level]
        else:
            mapping[cat] = path[-1]
    return mapping


def map_poi_categories(df, taxonomy_df, level):
    mapping = build_category_mapping(taxonomy_df, level)
    df = df.copy()
    df["category_level_" + str(level)] = df["primary_category"].map(mapping)
    return df


## Load overture categories
taxonomy = pd.read_csv("./data/prod/overture_categories.csv", sep=";")
taxonomy.columns = [c.strip() for c in taxonomy.columns]
taxonomy = taxonomy.map(lambda x: x.strip() if isinstance(x, str) else x)
taxonomy["Overture Taxonomy"] = taxonomy["Overture Taxonomy"].apply(transform_taxonomy)

level = 0
df_grouped_cat = map_poi_categories(filtered_df, taxonomy, level)

num_distinct_categories = df_grouped_cat["category_level_0"].nunique()
print("Number of distinct categories:", num_distinct_categories)

In [ ]:
df_grouped_cat.head()

In [ ]:
plt.hist(df["confidence"], bins=20, edgecolor="black")
plt.xlabel("Confidence")
plt.ylabel("Frequency")
plt.savefig("./outputs/new_figs/confidence_histogram.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
category_counts = df_grouped_cat["category_level_0"].value_counts()

category_counts.plot(kind="bar", figsize=(10, 6), edgecolor="black")
plt.xlabel("Category")
plt.ylabel("Number of POIs")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("./outputs/new_figs/category_distribution.pdf", format="pdf", bbox_inches="tight")
plt.show()

## Spatial Aggregation Example

This cell shows the code used to perform spatial aggregation and obtain data saved in ./data/prod/pre-process/traffic_cams_by_junc/

In [ ]:
grouped_df, agg_list, mapping = merge_data_by_junc(
    data = aggregated_timeseries, 
    metadata = traffic_cams_df_with_dirs,
    use_direction=True,
    offset=0,
    eps=50
)

In [ ]:
print(f"Number of junctions with EPS = 50m: {len(grouped_df.columns)}")

### Ablation

In [ ]:
grouped_df_eps_25, _, _ = merge_data_by_junc(
    data = aggregated_timeseries, 
    metadata = traffic_cams_df_with_dirs,
    use_direction=True,
    offset=0,
    eps=25
)

print(f"Number of junctions with EPS = 25m: {len(grouped_df_eps_25.columns)}")

In [ ]:
grouped_df_eps_100, _, _ = merge_data_by_junc(
    data = aggregated_timeseries, 
    metadata = traffic_cams_df_with_dirs,
    use_direction=True,
    offset=0,
    eps=100
)

print(f"Number of junctions with EPS = 100m: {len(grouped_df_eps_100.columns)}")

## Corr analysis

In [ ]:
corr_matrix = aggregated_timeseries.corr(method="pearson")

n = corr_matrix.shape[0]
upper_idx = np.triu_indices(n, k=1)          # k=1 skips the diagonal
pairwise_corrs = corr_matrix.values[upper_idx]

print(f"\nNumber of node pairs: {len(pairwise_corrs)}")
print(f"Pearson correlation  –  mean: {pairwise_corrs.mean():.4f} "
      f"| std: {pairwise_corrs.std():.4f} "
      f"| min: {pairwise_corrs.min():.4f} "
      f"| max: {pairwise_corrs.max():.4f}")

all_values = aggregated_timeseries.values.flatten()
all_values = all_values[~np.isnan(all_values)]   # remove NaNs

print(f"\nTraffic values  –  mean: {all_values.mean():.2f} "
      f"| std: {all_values.std():.2f} "
      f"| min: {all_values.min():.2f} "
      f"| max: {all_values.max():.2f}")

fig, ax = plt.subplots(figsize=(6, 4))

# Compute true probabilities: each bar = fraction of total pairs
counts, bin_edges = np.histogram(pairwise_corrs, bins=20, range=(-0.2, 1.0))
probs = counts / counts.sum()  # now each bar is a true probability, and they sum to 1

ax.bar(
    x=bin_edges[:-1],           # left edge of each bin
    height=probs,
    width=np.diff(bin_edges),   # bin width
    align="edge",
    color="steelblue",
    edgecolor="white"
)

ax.set_xlabel("Pearson correlation")
ax.set_ylabel("Probability")
ax.set_title("Distribution of inter-node correlations")
ax.set_xlim(-0.2, 1.0)
ax.set_ylim(0, None)

plt.tight_layout()
plt.show()

## Transition Probs analysis

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import osmnx as ox
import networkx as nx
import geopandas as gpd
from scripts.get_adj_matrix import map_lat_lon_to_osmnx_nodes
import networkx as nx


def prepare_camera_metadata(camera_df):
    filtered_cams = []
    
    for _, row in camera_df.iterrows():
        filtered_cams.append({
            "id_old": row['CAMERANAME'],
            "id": row['id'],
            "lat": row['LAT'],
            "lon": row['LON'],
            "veh_count": row['veh_count']
        })
    
    return filtered_cams


def get_road_distance_matrix(filtered_cams):
    graph_path = "./data/prod/pre-process/road_network/osmnx.graphml"
    G_road = ox.load_graphml(graph_path)
    cam_to_osmnx = map_lat_lon_to_osmnx_nodes(filtered_cams, G_road, lat_key="lat", lon_key="lon")
    nodes = [(cam['id_old'], {k:v for k, v in cam.items() if k != "id"}) for cam in filtered_cams]
    
    distance_matrix = {}
    
    for i in range(len(nodes)):
        for j in range(len(nodes)):
            id1, _ = nodes[i]
            id2, _ = nodes[j]
            node1, node2 = cam_to_osmnx[id1][0], cam_to_osmnx[id2][0]
            try:
                distance = nx.shortest_path_length(G_road, node1, node2, weight='length')
            except nx.NetworkXNoPath:
                raise ValueError(f"No path between camera {id1} and camera {id2} in the road graph.")
            distance = max(5, distance) # If two sensors are mapped to the same node, set a minimum distance of 10 meters
            
            distance_matrix[(id1, id2)] = distance
    
    return distance_matrix
            

def plot_road_distance_distribution(
    transition_df: pd.DataFrame,
    filtered_cams: list,
    min_count: int = 3,
) -> None:

    print("Computing road distance matrix...")
    distance_matrix = get_road_distance_matrix(filtered_cams)

    df = transition_df.copy()
    df["road_distance"] = df.apply(
        lambda r: distance_matrix.get((r["from"], r["to"]), np.nan),
        axis=1
    )

    df_filtered = df[df["count_ij"] >= min_count].dropna(subset=["road_distance"])
    x = df_filtered["road_distance"].values

    plt.figure(figsize=(6, 5))
    plt.hist(x, bins=30, edgecolor="white", color="steelblue")
    plt.xlabel("Road distance (m)")
    plt.ylabel("Count")
    plt.title(f"Road Distance Distribution\nn={len(df_filtered)} pairs (count_ij ≥ {min_count})")
    plt.tight_layout()
    plt.savefig("road_distance_distribution.png", dpi=150)
    plt.show()

In [ ]:
transition_df = pd.read_csv("./data/prod/pre-process/plate_hash/transition_probs_hour.csv")
filtered_cams = prepare_camera_metadata(traffic_cams_df_with_dirs)

In [ ]:
results = plot_road_distance_distribution(
    transition_df=transition_df,
    filtered_cams=filtered_cams,
    min_count=3
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def validate_rush_hour_signature(df: pd.DataFrame, min_count: int = 5) -> pd.DataFrame:
    """
    Aggregate mean count_ij by hour and weekday/weekend.
    Expected: clear morning (08:00) and evening (17:00-18:00) peaks on weekdays.
    """
    df_filtered = df[df["count_ij"] >= min_count]

    hourly = (
        df_filtered.groupby(["hour", "is_weekend"])
        .agg(total_count=("count_ij", "sum"))
        .reset_index()
    )
    wd = hourly[~hourly["is_weekend"]]
    we = hourly[hourly["is_weekend"]]
    print("\n--- Rush-hour signature ---")
    print(f"  Entries filtered (count_ij < {min_count}): {len(df) - len(df_filtered)}")
    print(f"  Weekday peak hour (count_ij): {wd.loc[wd['total_count'].idxmax(), 'hour']:02d}:00")
    print(f"  Weekend peak hour (count_ij): {we.loc[we['total_count'].idxmax(), 'hour']:02d}:00")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(wd["hour"], wd["total_count"], marker="o", ms=4, color="#2C7BB6", label="Weekday")
    ax.plot(we["hour"], we["total_count"], marker="s", ms=4, color="#D7191C", label="Weekend")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Mean count_ij")
    ax.set_title(f"Rush-hour signature (count_ij ≥ {min_count})")
    ax.set_xticks(range(0, 24, 3))
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("rush_hour_signature.png", dpi=150)
    plt.show()

    return hourly

In [ ]:
def load_dynamic_transitions(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["time_bin"])
    df["time_bin"] = pd.to_datetime(df["time_bin"].values, utc=True).tz_convert("Europe/Rome")
    df["hour"] = df["time_bin"].dt.hour
    df["day_of_week"] = df["time_bin"].dt.dayofweek       # 0=Mon, 6=Sun
    df["is_weekend"] = df["day_of_week"] >= 5
    return df

transition_df = load_dynamic_transitions("./data/prod/pre-process/plate_hash_by_junc/transition_probs_hour.csv")

_ = validate_rush_hour_signature(transition_df, min_count=3)

In [ ]:
def plot_outdegree_over_time(
    df: pd.DataFrame,
    min_count: int = 5,
    save_path: str = "outdegree_over_time.png",
) -> pd.DataFrame:
    """
    Plot the average out-degree per sensor across hours of the day
    for edges with count_ij >= min_count, weekday vs weekend.
    """

    od = (
        df[df["count_ij"] >= min_count]
        .groupby(["time_bin", "hour", "is_weekend", "from"])["to"]
        .size()
        .reset_index(name="out_degree")
        .groupby(["hour", "is_weekend"])["out_degree"]
        .mean()
        .reset_index(name="mean_out_degree")
    )

    wd = od[~od["is_weekend"]]
    we = od[od["is_weekend"]]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(wd["hour"], wd["mean_out_degree"], marker="o", ms=4,
            color="#2C7BB6", label="Weekday")
    ax.plot(we["hour"], we["mean_out_degree"], marker="s", ms=4,
            color="#D7191C", label="Weekend")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Mean out-degree per sensor")
    # ax.set_title(f"Average out-degree over time (count_ij ≥ {min_count})")
    ax.set_xticks(range(0, 24, 3))
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    return od

In [ ]:
_  = plot_outdegree_over_time(transition_df, min_count=3, save_path="./outputs/new_figs/outdegree_over_time.pdf")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


def analyze_matrix_sparsity(
    df: pd.DataFrame,
    n_sensors: int,
    min_count: int = 5,
    plot: bool = True,
    save_path: str = "sparsity_analysis.png",
) -> dict:
    """
    Quantify transition probability matrix sparsity along four axes:

    1. Structural density  — fraction of non-zero (i→j) pairs over all
                             possible directed pairs (n_sensors × (n_sensors-1)).
    2. Thresholded density — same but requiring count_ij >= min_count.
    3. Out-degree distribution — for each sensor, how many distinct
                                 downstream sensors it reaches.
    4. Temporal sparsity   — number of active edges per hourly time bin,
                             showing how the graph thins at night.

    Parameters
    ----------
    df         : dynamic transition probability DataFrame (with 'time_bin' parsed).
    n_sensors  : total number of sensors in the network (after cleaning).
    min_count  : reliability threshold for thresholded density.
    plot       : whether to produce diagnostic plots.
    save_path  : output path for the figure.
    """

    max_pairs = n_sensors * (n_sensors - 1)

    # ------------------------------------------------------------------ #
    # 1. Structural density (any observed transition counts)
    # ------------------------------------------------------------------ #
    observed_pairs = df[["from", "to"]].drop_duplicates()
    n_observed = len(observed_pairs)
    structural_density = n_observed / max_pairs

    # ------------------------------------------------------------------ #
    # 2. Thresholded density (only reliable transitions)
    # ------------------------------------------------------------------ #
    reliable_pairs = (
        df.groupby(["from", "to"])["count_ij"]
        .sum()
        .reset_index()
    )
    reliable_pairs = reliable_pairs[reliable_pairs["count_ij"] >= min_count]
    n_reliable = len(reliable_pairs)
    thresholded_density = n_reliable / max_pairs

    print("--- Matrix sparsity ---")
    print(f"  Total possible directed pairs:  {max_pairs}")
    print(f"  Observed pairs (any count):     {n_observed}  "
          f"→ structural density  = {structural_density:.4f} ({structural_density*100:.2f}%)")
    print(f"  Reliable pairs (≥{min_count} count):    {n_reliable}  "
          f"→ thresholded density = {thresholded_density:.4f} ({thresholded_density*100:.2f}%)")

    # ------------------------------------------------------------------ #
    # 3. Out-degree distribution
    # ------------------------------------------------------------------ #
    out_degree_all = (
        observed_pairs.groupby("from")["to"]
        .count()
        .reset_index(name="out_degree")
    )
    out_degree_reliable = (
        reliable_pairs.groupby("from")["to"]
        .count()
        .reset_index(name="out_degree")
    )

    print(f"\n  Out-degree (any count)     — "
          f"mean: {out_degree_all['out_degree'].mean():.2f}, "
          f"max: {out_degree_all['out_degree'].max()}, "
          f"min: {out_degree_all['out_degree'].min()}")
    print(f"  Out-degree (≥{min_count} count)     — "
          f"mean: {out_degree_reliable['out_degree'].mean():.2f}, "
          f"max: {out_degree_reliable['out_degree'].max()}, "
          f"min: {out_degree_reliable['out_degree'].min()}")

    # ------------------------------------------------------------------ #
    # 5. P_ij magnitude distribution among non-zero entries
    # ------------------------------------------------------------------ #
    pij_nonzero = df[df["P_ij"] > 0]["P_ij"]

    # ------------------------------------------------------------------ #
    # Plots
    # ------------------------------------------------------------------ #
    if plot:
        fig = plt.figure(figsize=(16, 9))
        gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
        c_all = "#2C7BB6"
        c_rel = "#D7191C"
        c_neu = "#555555"

        # Panel 1: structural vs thresholded density bar
        ax1 = fig.add_subplot(gs[0, 0])
        bars = ax1.bar(
            ["Structural\n(any count)", f"Thresholded\n(≥{min_count})"],
            [structural_density * 100, thresholded_density * 100],
            color=[c_all, c_rel], edgecolor="white", width=0.5
        )
        ax1.set_ylabel("Matrix density (%)")
        ax1.set_title("Structural vs thresholded density")
        for bar, val in zip(bars, [structural_density, thresholded_density]):
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.1,
                f"{val*100:.2f}%", ha="center", va="bottom", fontsize=10
            )

        # Panel 2: out-degree distribution (both thresholds)
        ax2 = fig.add_subplot(gs[0, 1])
        bins = range(0, max(out_degree_all["out_degree"].max(),
                            out_degree_reliable["out_degree"].max()) + 2)
        ax2.hist(out_degree_all["out_degree"], bins=bins, alpha=0.6,
                 color=c_all, edgecolor="white", label="Any count")
        ax2.hist(out_degree_reliable["out_degree"], bins=bins, alpha=0.6,
                 color=c_rel, edgecolor="white", label=f"≥{min_count} count")
        ax2.set_xlabel("Out-degree (number of reachable sensors)")
        ax2.set_ylabel("Number of sensors")
        ax2.set_title("Out-degree distribution")
        ax2.legend(fontsize=9)

        # Panel 4: P_ij magnitude distribution (log scale)
        ax4 = fig.add_subplot(gs[1, 0])
        ax4.hist(pij_nonzero, bins=60, color=c_neu, edgecolor="white")
        ax4.set_xlabel("P_ij (non-zero entries)")
        ax4.set_ylabel("Frequency")
        ax4.set_title("P_ij magnitude distribution")
        ax4.set_yscale("log")

        # Panel 5: cumulative density of P_ij
        ax5 = fig.add_subplot(gs[1, 1])
        sorted_pij = np.sort(pij_nonzero)
        cdf = np.arange(1, len(sorted_pij) + 1) / len(sorted_pij)
        ax5.plot(sorted_pij, cdf, color=c_neu, lw=1.5)
        ax5.axvline(0.01, color="orange", ls="--", lw=1.2, label="P_ij = 0.01")
        ax5.axvline(0.1,  color="red",    ls="--", lw=1.2, label="P_ij = 0.10")
        ax5.set_xlabel("P_ij")
        ax5.set_ylabel("Cumulative fraction of edges")
        ax5.set_title("CDF of non-zero P_ij values")
        ax5.legend(fontsize=9)

        plt.suptitle("Transition probability matrix — sparsity analysis",
                     fontsize=13, y=1.01)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"\nFigure saved to {save_path}")

    return {
        "max_pairs":            max_pairs,
        "n_observed":           n_observed,
        "n_reliable":           n_reliable,
        "structural_density":   structural_density,
        "thresholded_density":  thresholded_density,
        "out_degree_all":       out_degree_all,
        "out_degree_reliable":  out_degree_reliable
    }


df = pd.read_csv("./data/prod/pre-process/plate_hash/transition_probs.csv")
results = analyze_matrix_sparsity(df, n_sensors=66, min_count=3)